# EHR-RAG demo: one admission end to end

This notebook reads **cached run outputs only** (it makes no model or API calls):
header and evidence units → retrieval per section → prompt → report with citations →
citation check (NLI) → one perturbation example with a diff.

By default it follows the chain of the most recent runs:
`analyze-perturbation` → `run-pilot` → `select-targets` → `generate` → `retrieve`, plus the
latest `evaluate` run of that generation run. Set any run id below to override. A section
whose run does not exist yet is skipped with a message.

In [ ]:
import csv
import difflib
import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
except ImportError:  # plain Python (e.g. tests)
    Markdown = str
    display = print

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUNS = PROJECT_ROOT / "runs"

# Optional overrides (run ids under runs/); None = follow the latest chain.
GEN_RUN = None
EVAL_RUN = None
TARGETS_RUN = None
PILOT_RUN = None
ANALYSIS_RUN = None
HADM_ID = None   # admission to show; None = the admission of the first perturbation case (or the first report)

In [ ]:
def meta(run_id):
    p = RUNS / run_id / "run_meta.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


def latest(command, **match):
    """Latest run (by id) of `command` with status ok whose run_meta matches `match`."""
    if not RUNS.is_dir():
        return None
    ids = sorted(p.name for p in RUNS.iterdir() if p.is_dir())
    for rid in reversed(ids):
        m = meta(rid)
        if m.get("command") == command and m.get("status") == "ok" and all(m.get(k) == v for k, v in match.items()):
            return rid
    return None


ANALYSIS_RUN = ANALYSIS_RUN or latest("analyze-perturbation")
if ANALYSIS_RUN and not PILOT_RUN:
    PILOT_RUN = meta(ANALYSIS_RUN).get("pilot_run")
PILOT_RUN = PILOT_RUN or latest("run-pilot")
if PILOT_RUN and not TARGETS_RUN:
    TARGETS_RUN = meta(PILOT_RUN).get("targets_run")
if TARGETS_RUN and not GEN_RUN:
    GEN_RUN = meta(TARGETS_RUN).get("generation_run")
GEN_RUN = GEN_RUN or latest("generate")
RETRIEVE_RUN = meta(GEN_RUN).get("retrieval_run") if GEN_RUN else None
EVAL_RUN = EVAL_RUN or (latest("evaluate", generation_run=GEN_RUN) if GEN_RUN else None)

for name in ("RETRIEVE_RUN", "GEN_RUN", "EVAL_RUN", "TARGETS_RUN", "PILOT_RUN", "ANALYSIS_RUN"):
    print(f"{name:13} {globals()[name]}")
assert GEN_RUN, "no generate run found under runs/ (see RUNBOOK.md)"

In [ ]:
def read_json(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def read_csv(p):
    with Path(p).open(encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


cases = read_csv(RUNS / ANALYSIS_RUN / "perturbation_results.csv") if ANALYSIS_RUN else []
edit_cases = [c for c in cases if c["kind"] == "edit"]
if HADM_ID is None:
    if edit_cases:
        HADM_ID = int(edit_cases[0]["hadm_id"])
    else:
        HADM_ID = int(sorted((RUNS / GEN_RUN / "ehr_rag").iterdir())[0].name)
REPORT_DIR = RUNS / GEN_RUN / "ehr_rag" / str(HADM_ID)
shown = read_json(REPORT_DIR / "evidence_shown.json")
prompt = (REPORT_DIR / "prompt.txt").read_text(encoding="utf-8")
print("admission:", HADM_ID)

## 1. Patient header (not citable) and sample evidence units

In [ ]:
header = prompt.split("Patient header:\n", 1)[1].split("\n\nEvidence", 1)[0]
print(header)
print()
print(f"{len(shown['units'])} evidence units shown to the model; first 8:")
for u in shown["units"][:8]:
    print(f"[{u['unit_id']}] ({u['unit_type']}) {u['text']}")

## 2. Retrieval per section

In [ ]:
rfile = RUNS / RETRIEVE_RUN / "retrieval" / f"{HADM_ID}.json" if RETRIEVE_RUN else None
if rfile and rfile.exists():
    r = read_json(rfile)
    texts = {u["unit_id"]: u["text"] for u in r["evidence_units"]}
    print("primary diagnosis:", r["primary_dx"])
    for section, ids in r["sections"].items():
        print(f"\n--- {section}")
        for q in r["queries"][section]:
            print(f"  query: {q['query']!r}")
        for uid in ids:
            print(f"  [{uid}] {texts.get(uid, '(dropped by max_units_total)')}")
    print("\nevidence order:", r["evidence"])
else:
    print("retrieve run not available")

## 3. Prompt (exactly as sent)

In [ ]:
print(prompt)

## 4. Generated report with citations

In [ ]:
parsed_meta = read_json(REPORT_DIR / "parsed.json")
raw = (REPORT_DIR / parsed_meta["raw_file"]).read_text(encoding="utf-8")
print(f"format status: {parsed_meta['status']} (raw file: {parsed_meta['raw_file']})\n")
print(raw)

## 5. Citation check (NLI verifier)

In [ ]:
if EVAL_RUN:
    rows = [json.loads(l) for l in (RUNS / EVAL_RUN / "sentences.jsonl").read_text(encoding="utf-8").splitlines()
            if l.strip()]
    rows = [s for s in rows if s["condition"] == "ehr_rag" and s["hadm_id"] == HADM_ID]
    texts = {u["unit_id"]: u["text"] for u in shown["units"]}
    for s in rows:
        if s["is_abstention"]:
            print(f"{s['section']:<16} (abstention) {s['sentence']}")
            continue
        verdict = "SUPPORTED" if s["supported"] else "not supported"
        print(f"{s['section']:<16} {verdict:<14} NLI={s['nli_label']:<18} {s['sentence']}")
        for uid in s["cited_ids"]:
            mark = " (irrelevant)" if uid in s["irrelevant_ids"] else ""
            print(f"{'':<32}[{uid}]{mark} {texts[uid]}")
        for uid in s["invalid_ids"]:
            print(f"{'':<32}[{uid}] <not in evidence: fabricated id>")
    display(Markdown((RUNS / EVAL_RUN / "metrics_summary.md").read_text(encoding="utf-8")))
else:
    print("evaluate run not available")

## 6. One perturbation example (frozen retrieval) with a diff

In [ ]:
example = next((c for c in edit_cases if int(c["hadm_id"]) == HADM_ID), edit_cases[0] if edit_cases else None)
if example and PILOT_RUN:
    print(f"case {example['case_id']}: {example['edit_type']} on [{example['unit_id']}] -> automated outcome: "
          f"{example['auto_outcome']} (provisional)\n")
    print("original sentence:", example["original_sentence"])
    print("evidence before  :", example["original_text"])
    print("evidence after   :", example["edited_text"])
    case_dir = RUNS / PILOT_RUN / "cases" / example["case_id"]
    case_raw = (case_dir / read_json(case_dir / "parsed.json")["raw_file"]).read_text(encoding="utf-8")
    orig_dir = RUNS / GEN_RUN / "ehr_rag" / example["hadm_id"]
    orig_raw = (orig_dir / read_json(orig_dir / "parsed.json")["raw_file"]).read_text(encoding="utf-8")
    print("\nreport diff (original -> regenerated):")
    print("\n".join(difflib.unified_diff(orig_raw.splitlines(), case_raw.splitlines(), "original",
                                         example["case_id"], lineterm="")) or "(no change)")
    display(Markdown((RUNS / ANALYSIS_RUN / "perturbation_summary.md").read_text(encoding="utf-8")))
else:
    print("perturbation runs not available")